# Part 2 - Data Visualization using dimensionality reduction

To succesfully learn from your data, you need to know it.
Dimension-reduction is a great tool to help analyze high dimensional data.
It can help us to see the data in two or three dimensions and analyze clusters or structures.
It can also be used to reduce the number of dimensions before processing the data using machine learning algorithms.
This can greatly decrease the computational load optimally without losing much information about the data.
In these tasks, you will learn how to use dimensionality reduction techniques to visualize the GalaxyMNIST dataset.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch, gzip, requests

# machine learning metrics
from sklearn import metrics

# performance metrics
from tqdm import tqdm
import time

from collections import Counter
from itertools import chain

# dataset
from galaxy_mnist import GalaxyMNIST

download_data = [
    {"url": "https://www.dropbox.com/scl/fi/l4ap5eq7z49l27dnfw076/train_dataset.hdf5.gz?rlkey=gjfrrrouoah0hbydbkpfx66as&st=tw6znd55&dl=1", "file": "train_dataset.hdf5",},
    {"url": "https://www.dropbox.com/scl/fi/ty43vhrrpa8cf8azn3awn/test_dataset.hdf5.gz?rlkey=jmlkzjiguhe271nbuzpa0njna&st=5oco8ika&dl=1", "file": "test_dataset.hdf5",},
]

root = Path("raw_galaxy_mnist")
raw = root / "GalaxyMNIST" / "raw"
raw.mkdir(parents=True, exist_ok=True)
for d in download_data:
    p = raw / d["file"]
    if not p.exists():
        r = requests.get(d["url"], timeout=60)
        p.write_bytes(gzip.decompress(r.content))

dataset = GalaxyMNIST(root=root, train=True)
val_dataset = GalaxyMNIST(root=root, train=False)

# Extract data and targets
raw_images, labels = dataset.data, dataset.targets
raw_test_images, test_labels = val_dataset.data, val_dataset.targets
num_classes = len(dataset.classes)

## 2.1 Reducing Dimensions to Reduce Computational Cost


PCA scales cubically with the number of dimensions because a matrix of size `n_features x n_features` needs to be diagonalized.

- How many entries does this matrix have when using full resolution GalaxyMNIST images?

To ease the burden on your CPU, you can down-scale the data either by transforming the images to gray-scale, down-sampling the images to 32x32 resolution or doing both. We recommend going to gray-scale for powerful/newer CPUs and doing both for less powerful ones.

- Implement one of these options. You can use torchvision transforms.
- By what factor do the methods reduce the number of features and what is the expected speedup?


The full feature dimension is $3 \cdot 64 \cdot 64 = 12288$ leading to a matrix size of $12288^2 \approx 150.000.000$ entries.


In [2]:
# Standardize training data
raw_images_float = raw_images.float() / 255.0
train_mean = raw_images_float.mean(dim=(0, 2, 3)).view(1, -1, 1, 1)
train_std = raw_images_float.std(dim=(0, 2, 3)).view(1, -1, 1, 1)
images = (raw_images_float - train_mean) / train_std

## 2.2 Principal Component Analysis (PCA)

### 2.2.1 PCA Algorithm

- Implement the PCA algorithm. It should return the transformed data points, all principal components (eigenvectors) and all eigenvalues. Use methods like eigh for the diagonalization from numpy, scipy or torch.


### 2.2.2 PCA Visualization

PCA can be used to create visualizations as well as analyze variance in the data. Use your PCA implementation to create the following plots:

- So-called scree plot, where you plot the eigenvalues against the number of components (use all eigenvalues).
- Explained variance plot, the variance explained by all components up to the n-th component divided by the total variance (use all eigenvalues).
- Dimension reduction plots. Use the data projected onto the first and second principal components to create a scatter plot. Color the points according to their true labels. Can you see which classes similar to each other from the plot? Bonus: Create a 3D plot with the first three principal components.
- Visualize the first few principal components (eigenvectors) as images. You can reshape the components to the original image dimensions and plot them.

**\*Bonus Note**: If you are unsure whether your implementation is correct, you may compare the principal components obtained by your implementation, $\{v_i\}_i$, with those obtained by the [scikit-learn PCA implementation](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html), $\{u_i\}_i$.\*\*

- _Are they completely different? Then your implementation is likely incorrect._
- _Are they equal? Then your implementation is likely correct._
- _Do some differ only in sign, i.e., $v_i = -u_i$? Then your implementation is probably correct. But why do they differ in sign?_

**_This note is intended as a sanity check and is neither necessary nor mandatory._**


## 2.3 UMAP

Two more advanced techniques for dimensionality reduction are t-SNE and UMAP. You will now try UMAP on the GalaxyMNIST dataset.


- Briefly read about UMAP. Would a different dataset normalization yield different visualizations?
- Use UMAP to visualize the GalaxyMNIST dataset. Use the umap package, e.g. `from umap import UMAP` rather than implementing it yourself. Plot the first and second components of the UMAP projection against each other. Color the points according to their true labels.
- Play with the hyperparameters and see how they affect the visualization.


## 2.4 Identifying Regions

In your UMAP or t-SNE plot, you can probably see some specific regions.

- Choose at least two regions, identify some samples inside them plot the original images.
- What do images in the same region share?
- Bonus: Create a large image of images in the latent space. This can be done by finding the nearest neighbors in the latent space and plotting the images in a grid.
